In [5]:
#imports

from typing import Optional, Tuple, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

In [6]:
"""
Swim-UNet (Swin-UNet style) segmentation model with a ResNet backbone encoder.

This file defines a PyTorch module `SwimUNetResNet`:
- Encoder: ResNet (resnet18 / resnet34 / resnet50) from torchvision (optionally pretrained).
- Bottleneck + decoder: lightweight Transformer blocks (windowless Multi-Head Self-Attention
  applied on flattened HW tokens) interleaved with upsampling and conv blocks.
- Skip connections from ResNet stages to decoder (UNet-style).

Notes:
- This implementation aims to provide a compact, easy-to-read model combining a ResNet
  encoder and Transformer-based decoder (a "Swin-like" design). It intentionally uses
  standard MultiheadAttention rather than a full windowed Swin implementation for
  simplicity and portability.
- Requires: torch, torchvision
"""



# -------------------------
# Utility blocks
# -------------------------
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Conv3x3GNReLU(nn.Module):
    """Alternative block if you prefer GroupNorm — not used by default but available."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.gn = nn.GroupNorm(8, out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.gn(self.conv(x)))


# -------------------------
# Transformer-style block
# -------------------------
class TransformerBlock2D(nn.Module):
    """
    Lightweight Transformer block that operates on spatial tokens.
    - Flattens HxW -> N tokens
    - Applies LayerNorm -> MHSA -> Dropout -> Residual
    - Then MLP (FFN) with GELU and residual
    """
    def __init__(self, in_channels: int, num_heads: int = 8, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(in_channels)
        self.attn = nn.MultiheadAttention(embed_dim=in_channels, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(in_channels)
        hidden_dim = int(in_channels * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, in_channels),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, C, H, W)
        returns: (B, C, H, W)
        """
        B, C, H, W = x.shape
        n_tokens = H * W
        # flatten to (B, N, C)
        tokens = x.view(B, C, n_tokens).permute(0, 2, 1)  # (B, N, C)
        # Pre-attn norm
        t = self.norm1(tokens)
        # MultiHeadAttention expects (B, N, C) if batch_first=True
        attn_out, _ = self.attn(t, t, t, need_weights=False)
        tokens = tokens + self.drop1(attn_out)  # Residual
        # MLP
        tokens = tokens + self.mlp(self.norm2(tokens))
        # back to (B, C, H, W)
        out = tokens.permute(0, 2, 1).contiguous().view(B, C, H, W)
        return out


# -------------------------
# Decoder block combining upsample -> concat skip -> conv -> transformer
# -------------------------
class DecoderBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, use_transformer: bool = True, attn_heads: int = 8):
        """
        in_ch: channels from previous decoder (or bottleneck)
        skip_ch: channels from encoder skip (0 if none)
        out_ch: output channels after this block
        """
        super().__init__()
        self.use_skip = skip_ch > 0
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        mid_ch = out_ch
        total_in = in_ch + skip_ch if self.use_skip else in_ch
        # conv block to fuse the concatenated features
        self.conv1 = ConvBNReLU(total_in, mid_ch)
        self.conv2 = ConvBNReLU(mid_ch, out_ch)
        # optional transformer to refine features spatially
        self.transformer = TransformerBlock2D(out_ch, num_heads=attn_heads) if use_transformer else None

    def forward(self, x: torch.Tensor, skip: Optional[torch.Tensor]) -> torch.Tensor:
        x = self.up(x)
        if self.use_skip and skip is not None:
            # if spatial sizes mismatch by 1 due to odd dims, center-crop or pad
            if x.shape[-2:] != skip.shape[-2:]:
                # simple resize skip to x spatial dims
                skip = F.interpolate(skip, size=x.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        if self.transformer is not None:
            x = self.transformer(x)
        return x


# -------------------------
# ResNet encoder wrapper
# -------------------------
class ResNetEncoder(nn.Module):
    def __init__(self, backbone: str = "resnet34", pretrained: bool = True):
        super().__init__()
        backbone = backbone.lower()
        if backbone == "resnet18":
            res = models.resnet18(pretrained=pretrained)
            feats = [64, 64, 128, 256, 512]
        elif backbone == "resnet34":
            res = models.resnet34(pretrained=pretrained)
            feats = [64, 64, 128, 256, 512]
        elif backbone == "resnet50":
            res = models.resnet50(pretrained=pretrained)
            feats = [64, 256, 512, 1024, 2048]
        else:
            raise ValueError(f"Unsupported backbone {backbone}. Choose resnet18/34/50.")

        # Using the layers from torchvision ResNet
        self.conv1 = res.conv1    # /2
        self.bn1 = res.bn1
        self.relu = res.relu
        self.maxpool = res.maxpool  # /2 (total /4 so far)

        self.layer1 = res.layer1  # typically keeps /4 resolution
        self.layer2 = res.layer2  # /8
        self.layer3 = res.layer3  # /16
        self.layer4 = res.layer4  # /32

        self.feature_channels = feats  # [conv1_out, layer1_out, layer2_out, layer3_out, layer4_out]

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        returns features at multiple scales for skip connections:
        - x0: after initial conv1 (before maxpool)  -> high-res (usually /2)
        - x1: after layer1 -> /4
        - x2: after layer2 -> /8
        - x3: after layer3 -> /16
        - x4: after layer4 -> /32 (deepest)
        """
        x0 = self.relu(self.bn1(self.conv1(x)))  # (B, c0, H/2, W/2)
        x1 = self.maxpool(x0)                    # (B, c0, H/4, W/4)  -- often used as input to layer1
        x1 = self.layer1(x1)                     # (B, c1, H/4, W/4)
        x2 = self.layer2(x1)                     # (B, c2, H/8, W/8)
        x3 = self.layer3(x2)                     # (B, c3, H/16, W/16)
        x4 = self.layer4(x3)                     # (B, c4, H/32, W/32)
        return x0, x1, x2, x3, x4


# -------------------------
# Swim-UNet model combining encoder + transformer-style decoder
# -------------------------
class SwimUNetResNet(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        num_classes: int = 1,
        backbone: str = "resnet34",
        pretrained_backbone: bool = True,
        decoder_channels: Optional[List[int]] = None,
        use_transformer_in_decoder: bool = True,
        transformer_heads: int = 8,
    ):
        """
        decoder_channels: list of channels for decoder stages from deep->shallow (len 4 typically).
        Example: [512, 256, 128, 64]
        """
        super().__init__()
        self.encoder = ResNetEncoder(backbone=backbone, pretrained=pretrained_backbone)
        enc_ch = self.encoder.feature_channels  # [c0, c1, c2, c3, c4]

        # We will build a 4-stage decoder (matching encoder layers 4->3->2->1)
        if decoder_channels is None:
            # choose sizes based on typical UNet design and backbone depth
            # pick decoder channels to be symmetric-ish with encoder
            decoder_channels = [enc_ch[-1] // 1, enc_ch[-2] // 1, enc_ch[-3] // 1, enc_ch[-4] // 1]

        assert len(decoder_channels) == 4, "decoder_channels must be length 4"

        # if input channels don't match encoder expected (resnet conv1), adapt with conv
        # ResNet conv1 usually expects 3 channels. If in_channels != 3, create adapter
        if in_channels != 3:
            self.input_adapter = nn.Conv2d(in_channels, 3, kernel_size=1)
        else:
            self.input_adapter = None

        # Bottleneck projection (optional) to decoder channel size
        self.bottleneck_proj = nn.Conv2d(enc_ch[-1], decoder_channels[0], kernel_size=1)

        # Decoder blocks: deep -> shallow
        self.decoder4 = DecoderBlock(in_ch=decoder_channels[0], skip_ch=enc_ch[-2], out_ch=decoder_channels[1],
                                     use_transformer=use_transformer_in_decoder, attn_heads=transformer_heads)
        self.decoder3 = DecoderBlock(in_ch=decoder_channels[1], skip_ch=enc_ch[-3], out_ch=decoder_channels[2],
                                     use_transformer=use_transformer_in_decoder, attn_heads=transformer_heads)
        self.decoder2 = DecoderBlock(in_ch=decoder_channels[2], skip_ch=enc_ch[-4], out_ch=decoder_channels[3],
                                     use_transformer=use_transformer_in_decoder, attn_heads=transformer_heads)
        # final decoder block to reach near-input resolution; skip from x0 (conv1 output)
        # Note: x0 is /2 resolution; after decoder2 upsample we are at /4 -> we can upsample twice to reach /2
        # So the final block will not have a skip by default from x0; we'll concat x0 after upsampling once more.
        self.final_up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.final_conv = nn.Sequential(
            ConvBNReLU(decoder_channels[3], decoder_channels[3]),
            ConvBNReLU(decoder_channels[3], decoder_channels[3] // 2 if decoder_channels[3] // 2 > 0 else decoder_channels[3]),
        )

        out_ch = max(1, decoder_channels[3] // 2)
        self.classifier = nn.Conv2d(out_ch, num_classes, kernel_size=1)

        # initialize classifier bias for better training stability (optional)
        # nn.init.constant_(self.classifier.bias, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.input_adapter:
            x = self.input_adapter(x)
        # Encoder
        x0, x1, x2, x3, x4 = self.encoder(x)  # x0: after conv1 (/2), x1:/4, x2:/8, x3:/16, x4:/32
        # Bottleneck projection
        bottleneck = self.bottleneck_proj(x4)  # reduce/align channels
        # Decoder pipeline
        d4 = self.decoder4(bottleneck, x3)  # now at /16 -> upsample -> /16->/8? (each decoder upsamples by 2)
        d3 = self.decoder3(d4, x2)          # /8->/4
        d2 = self.decoder2(d3, x1)          # /4->/2
        # Bring to original-ish resolution: upsample once more, concat with x0, convs
        up = self.final_up(d2)              # /2 resolution (same as x0)
        # Ensure spatial sizes match
        if up.shape[-2:] != x0.shape[-2:]:
            x0_resized = F.interpolate(x0, size=up.shape[-2:], mode='bilinear', align_corners=False)
        else:
            x0_resized = x0
        fused = up + x0_resized  # element-wise fusion; could also concat
        out = self.final_conv(fused)
        logits = self.classifier(out)
        # if segmentation with probabilities: caller can apply sigmoid/softmax
        return logits



In [ ]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ---------------------------
# 1. Dataset loader
# ---------------------------
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None, mask_transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        return image, mask


# ---------------------------
# 2. Define transforms
# ---------------------------
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

mask_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])


# ---------------------------
# 3. Automatically split dataset
# ---------------------------
def get_dataset_splits(image_dir, mask_dir, val_ratio=0.15, test_ratio=0.15, seed=42):
    image_paths = sorted([os.path.join(image_dir, x) for x in os.listdir(image_dir)])
    mask_paths = sorted([os.path.join(mask_dir, x) for x in os.listdir(mask_dir)])
    assert len(image_paths) == len(mask_paths), "Mismatch in number of images and masks!"
    
    train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
        image_paths, mask_paths, test_size=(val_ratio + test_ratio), random_state=seed
    )
    val_size = test_ratio / (val_ratio + test_ratio)
    val_imgs, test_imgs, val_masks, test_masks = train_test_split(
        temp_imgs, temp_masks, test_size=val_size, random_state=seed
    )
    return (train_imgs, train_masks), (val_imgs, val_masks), (test_imgs, test_masks)


# ---------------------------
# 4. Create datasets and loaders
# ---------------------------
(train_imgs, train_masks), (val_imgs, val_masks), (test_imgs, test_masks) = get_dataset_splits(
    image_dir="/kaggle/input/covid19-ct-scan-lesion-segmentation-dataset/frames",
    mask_dir="/kaggle/input/covid19-ct-scan-lesion-segmentation-dataset/masks"
)

train_dataset = SegmentationDataset(train_imgs, train_masks, train_transform, mask_transform)
val_dataset = SegmentationDataset(val_imgs, val_masks, train_transform, mask_transform)
test_dataset = SegmentationDataset(test_imgs, test_masks, train_transform, mask_transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

print(f"✅ Dataset split: Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}")


# ---------------------------
# 5. Model, Loss, Optimizer
# ---------------------------
# from model import SwimUNetResNet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SwimUNetResNet(in_channels=3, num_classes=1, backbone="resnet34", pretrained_backbone=False).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


# ---------------------------
# 6. Metrics (Accuracy + Dice)
# ---------------------------
def dice_coefficient(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    target = (target > 0.5).float()
    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2. * intersection + smooth) / (union + smooth)
    return dice.mean().item()

def pixel_accuracy(pred, target):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    target = (target > 0.5).float()
    correct = (pred == target).float().sum()
    total = torch.numel(pred)
    return (correct / total).item()


# ---------------------------
# 7. Training and Validation
# ---------------------------
def train_one_epoch(loader, model, optimizer, loss_fn, device):
    model.train()
    epoch_loss, epoch_acc, epoch_dice = 0.0, 0.0, 0.0
    for imgs, masks in tqdm(loader, desc="Training", leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        masks = torch.nn.functional.interpolate(masks, size=preds.shape[2:], mode='nearest')
        loss = loss_fn(preds, masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += pixel_accuracy(preds, masks)
        epoch_dice += dice_coefficient(preds, masks)
    return epoch_loss / len(loader), epoch_acc / len(loader), epoch_dice / len(loader)


def evaluate(loader, model, loss_fn, device):
    model.eval()
    val_loss, val_acc, val_dice = 0.0, 0.0, 0.0
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Validation", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            masks = torch.nn.functional.interpolate(masks, size=preds.shape[2:], mode='nearest')
            loss = loss_fn(preds, masks)
            val_loss += loss.item()
            val_acc += pixel_accuracy(preds, masks)
            val_dice += dice_coefficient(preds, masks)
    return val_loss / len(loader), val_acc / len(loader), val_dice / len(loader)


# ---------------------------
# 8. Training Loop
# ---------------------------
num_epochs = 20
best_val_loss = float("inf")

for epoch in range(num_epochs):
    train_loss, train_acc, train_dice = train_one_epoch(train_loader, model, optimizer, criterion, device)
    val_loss, val_acc, val_dice = evaluate(val_loader, model, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train -> Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Dice: {train_dice:.4f}")
    print(f"Val   -> Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, Dice: {val_dice:.4f}\n")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_swim_unet_resnet.pth")
        print("✅ Saved best model")

print("Training Complete!")


# ---------------------------
# 9. Testing
# ---------------------------
# def test_model(model, loader, save_dir="outputs"):
#     os.makedirs(save_dir, exist_ok=True)
#     model.eval()
#     total_acc, total_dice = 0.0, 0.0
#     with torch.no_grad():
#         for i, (imgs, masks) in enumerate(tqdm(loader, desc="Testing")):
#             imgs, masks = imgs.to(device), masks.to(device)
#             preds = model(imgs)
#             masks = torch.nn.functional.interpolate(masks, size=preds.shape[2:], mode='nearest')
#             total_acc += pixel_accuracy(preds, masks)
#             total_dice += dice_coefficient(preds, masks)
#             preds = torch.sigmoid(preds)
#             preds = (preds > 0.5).float()
#             for j in range(preds.size(0)):
#                 pred_mask = preds[j].cpu()
#                 out_img = transforms.ToPILImage()(pred_mask.squeeze(0))
#                 out_img.save(os.path.join(save_dir, f"pred_{i*loader.batch_size + j}.png"))
#     avg_acc = total_acc / len(loader)
#     avg_dice = total_dice / len(loader)
#     print(f"✅ Testing complete! Avg Acc: {avg_acc:.4f} | Avg Dice: {avg_dice:.4f}")
#     print("Predicted masks saved to:", save_dir)


# Example usage:
# test_model(model, test_loader)


✅ Dataset split: Train=1910 | Val=409 | Test=410


Epoch [1/20]
Train -> Loss: 0.3404, Acc: 0.9790, Dice: 0.0958
Val   -> Loss: 0.2627, Acc: 0.9837, Dice: 0.2244

✅ Saved best model


Epoch [2/20]
Train -> Loss: 0.1962, Acc: 0.9874, Dice: 0.2779
Val   -> Loss: 0.1585, Acc: 0.9877, Dice: 0.3111

✅ Saved best model


Epoch [3/20]
Train -> Loss: 0.1193, Acc: 0.9899, Dice: 0.3742
Val   -> Loss: 0.0955, Acc: 0.9892, Dice: 0.4722

✅ Saved best model


Epoch [4/20]
Train -> Loss: 0.0791, Acc: 0.9909, Dice: 0.4325
Val   -> Loss: 0.0687, Acc: 0.9908, Dice: 0.4987

✅ Saved best model


...

Epoch [7/20]
Train -> Loss: 0.0322, Acc: 0.9936, Dice: 0.5767
Val   -> Loss: 0.0316, Acc: 0.9927, Dice: 0.5551

✅ Saved best model


Epoch [8/20]
Train -> Loss: 0.0263, Acc: 0.9940, Dice: 0.5978
Val   -> Loss: 0.0281, Acc: 0.9926, Dice: 0.6247

✅ Saved best model


Epoch [9/20]
Train -> Loss: 0.0228, Acc: 0.9942, Dice: 0.6172
Val   -> Loss: 0.0243, Acc: 0.9930, Dice: 0.6262

✅ Saved best model


Epoch [10/20]
Train -> Loss: 0.0197, Acc: 0.9947, Dice: 0.6429
Val   -> Loss: 0.0226, Acc: 0.9933, Dice: 0.6317

✅ Saved best model


Training:   5%|▍         | 23/478 [00:06<02:01,  3.74it/s]

In [ ]:
def test_model(model, loader, save_dir="outputs"):
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    total_acc, total_dice = 0.0, 0.0
    with torch.no_grad():
        for i, (imgs, masks) in enumerate(tqdm(loader, desc="Testing")):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            masks = torch.nn.functional.interpolate(masks, size=preds.shape[2:], mode='nearest')
            total_acc += pixel_accuracy(preds, masks)
            total_dice += dice_coefficient(preds, masks)
            preds = torch.sigmoid(preds)
            preds = (preds > 0.5).float()
            for j in range(preds.size(0)):
                pred_mask = preds[j].cpu()
                out_img = transforms.ToPILImage()(pred_mask.squeeze(0))
                out_img.save(os.path.join(save_dir, f"pred_{i*loader.batch_size + j}.png"))
    avg_acc = total_acc / len(loader)
    avg_dice = total_dice / len(loader)
    print(f"✅ Testing complete! Avg Acc: {avg_acc:.4f} | Avg Dice: {avg_dice:.4f}")
    print("Predicted masks saved to:", save_dir)
test_model(model, test_loader)